In [1]:
import pandas as pd

In [ ]:
def preprocess_peptide_dataset(csv_path, min_samples_per_cell_line=10, exclude_csv_path=None):
    # 1. Loading the dataset
    print(f"Loading the dataset from {csv_path}...")
    df = pd.read_csv(csv_path)
    print(f"Initial dataset size: {len(df)} rows")
    
    # 2. Keep only the columns we need
    df = df[['sequence', 'is_cpp', 'cell_line']].copy()
    
    # 3. Standardizing sequences
    print("\nStandardizing sequences...")
    df['sequence'] = df['sequence'].astype(str).str.upper()
    allowed_chars = set("ACDEFGHIKLMNPQRSTVWYXZBOU")
    
    def is_valid_sequence(seq, allowed_set):
        if not isinstance(seq, str) or not seq:
            return False
        return all(char in allowed_set for char in seq)
    
    # Sequence filtering
    df['is_valid'] = df['sequence'].apply(lambda x: is_valid_sequence(x, allowed_chars))
    df_filtered = df[df['is_valid']].copy()
    df_filtered = df_filtered.drop(columns=['is_valid'])
    
    print(f"After filtering for valid characters: {len(df_filtered)} rows")
    print(f"Filtered: {len(df) - len(df_filtered)} rows")
    
    # removal of matching sequences
    if exclude_csv_path:
        print(f"\nRemoval of sequences that match {exclude_csv_path}...")
        df_exclude = pd.read_csv(exclude_csv_path)

        if 'sequence' in df_exclude.columns:
            exclude_sequences = set(df_exclude['sequence'].astype(str))
            
            count_before = len(df_filtered)
            df_filtered = df_filtered[~df_filtered['sequence'].isin(exclude_sequences)].copy()
            excluded_count = count_before - len(df_filtered)

            print(f"Matches found and removed: {excluded_count}")
            print(f"Remaining after removal: {len(df_filtered)} rows")

        else:
            print(f"ATTENTION: The file {exclude_csv_path} does not contain a ‘sequence’ column. Skipping the step.")
    
    # 4. CPP Statistics
    print("\nCPP Statistics:")
    cpp_counts = df_filtered['is_cpp'].value_counts()
    print(cpp_counts)

    # Removing rows containing NaN in cell_line
    nan_count = df_filtered['cell_line'].isna().sum()
    if nan_count > 0:
        print(f"\n{nan_count} rows with no value in the cell_line column have been found. Removing them.")
        df_filtered = df_filtered.dropna(subset=['cell_line']).copy()
        print(f"Remaining after removal: {len(df_filtered)} rows")

    # 5. CPP Statistics
    print("\nCPP Statistics:")
    cpp_counts = df_filtered['is_cpp'].value_counts()
    print(cpp_counts)

    # 6.  Filter by is_cpp
    print("\nFilter by is_cpp...")
    cpp_df = df_filtered[df_filtered['is_cpp'] == 1].copy()
    print(f"CPP: {len(cpp_df)}")
    
    # 7.  CPP distribution across cell lines BEFORE the merger of rare

    # Counting samples for each cell line
    cell_line_counts = cpp_df['cell_line'].value_counts()
    print("\nDistribution by cell line before merging:")
    for cell_line, count in cell_line_counts.items():
        print(f" {cell_line}: {count} samples")
    
    # Identifying rare cell lines
    print(f"\nProcessing cell lines (minimum samples:  {min_samples_per_cell_line})...")
    rare_cell_lines = cell_line_counts[cell_line_counts < min_samples_per_cell_line].index.tolist()
    
    if 'unknown' in rare_cell_lines:
        rare_cell_lines.remove('unknown')

    if rare_cell_lines:
        print(f"\nCell lines with fewer than < {min_samples_per_cell_line} samples (will be grouped under 'other')")
        for cl in rare_cell_lines:
            print(f"  {cl}: {cell_line_counts[cl]} samples")
        
        # Replace rare cell lines with "other"
        cpp_df.loc[df_filtered['cell_line'].isin(rare_cell_lines), 'cell_line'] = 'other'
    
    # Final distribution
    final_cell_line_counts = cpp_df['cell_line'].value_counts()
    print("\nDistribution by cell line:")
    for cell_line, count in final_cell_line_counts.items():
        print(f"  {cell_line}: {count} samples")
    
    return cpp_df

def perform_oversampling(df, target_samples):
    
    grouped = df.groupby('cell_line')
    
    resampled_dfs = []
    for name, group in grouped:
        if name == 'unknown':
            print(f"The '{name}' class will not be oversampled (size:) {len(group)})")
            resampled_dfs.append(group)
            continue
    
        if len(group) < target_samples:
            # Duplicate random strings to reach the desired number
            resampled_group = group.sample(n=target_samples, replace=True, random_state=42)
            resampled_dfs.append(resampled_group)
        else:
            resampled_dfs.append(group)
            
    df_resampled = pd.concat(resampled_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
    return df_resampled

def format_sequences_for_protgpt2(df, seq_column='sequence'):
    """
    Converts sequences to the ProtGPT2 format:
    - Adds <|endoftext|> at the beginning
    - Adds line breaks every 60 amino acids
    
    Args:
        df: DataFrame containing sequences
        seq_column: name of the column containing the sequences
    
    Returns:
        DataFrame containing the converted sequences
    """
    df = df.copy()
    
    def format_sequence(seq):
        formatted = "<|endoftext|>"
        
        for i in range(0, len(seq), 60):
            formatted += "\n" + seq[i:i+60]
        
        formatted += "\n<|endoftext|>"
        
        return formatted
    
    df[seq_column] = df[seq_column].apply(format_sequence)
    
    return df

In [ ]:
path = '../data/active_sampling/peptides_without_classifier_train_rows.csv'
exclude_path = '../data/active_sampling/train_rows_for_classifier.csv'

In [4]:
processed_df = preprocess_peptide_dataset(path, min_samples_per_cell_line=10, exclude_csv_path=exclude_path)

Loading the dataset from ../active_sampling/peptides_without_train_rows_for_classifier.csv...
Initial dataset size: 2422 rows

Standardizing sequences...
After filtering for valid characters: 2182 rows
Filtered: 240 rows

Removal of sequences that match ../active_sampling/train_rows_for_classifier.csv...
Matches found and removed: 11
Remaining after removal: 2171 rows

CPP Statistics:
is_cpp
True     1118
False    1053
Name: count, dtype: int64

1522 rows with no value in the cell_line column have been found. Removing them.
Remaining after removal: 649 rows

CPP Statistics:
is_cpp
True     559
False     90
Name: count, dtype: int64

Filter by is_cpp...
CPP: 559

Distribution by cell line before merging:
 HeLa cells: 142 samples
 NIH-3T3 cells: 48 samples
 A549 cells: 34 samples
 CHO-K1 cells: 28 samples
 MDA-MB-231 cells: 27 samples
 CHO cells: 26 samples
 MCF7 cells: 23 samples
 HaCaT cells: 22 samples
 HEK293 cells: 17 samples
 U87 cells: 15 samples
 Human bowes melanoma cells: 12 sa

In [5]:
cpp_df = processed_df[processed_df['is_cpp'] == 1].copy()

# Determining the target number of samples
target_count = cpp_df[cpp_df['cell_line'] != 'unknown']['cell_line'].value_counts().max()
print(f"\nOversampling minor classes to {target_count} samples is in progress...")

# Applying oversampling
df_train_balanced = perform_oversampling(cpp_df, target_count)

print("\nDataset size after oversampling:")
print(df_train_balanced['cell_line'].value_counts())
print(f"Total size of the training dataset: {len(df_train_balanced)} rows")


Oversampling minor classes to 165 samples is in progress...

Dataset size after oversampling:
cell_line
MDA-MB-231 cells              165
A549 cells                    165
CHO-K1 cells                  165
U87 cells                     165
Human bowes melanoma cells    165
CHO cells                     165
MCF7 cells                    165
HeLa cells                    165
HEK293 cells                  165
NIH-3T3 cells                 165
HaCaT cells                   165
other                         165
Name: count, dtype: int64
Total size of the training dataset: 1980 rows


In [ ]:
cpp_df.to_csv('../data/processed/generation_train_dataset.csv', index=False)

In [7]:
# Formatting for ProtGPT2
df_train_balanced = format_sequences_for_protgpt2(df_train_balanced)

In [ ]:
df_train_balanced.to_csv('../data/processed/generation_train_dataset_formatted.csv', index=False)